# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toBESkiii/FlyRank_AI_Intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install --upgrade duckdb

import os
import duckdb
from google.colab import userdata

# Read the token securely from Colab Secrets
hugging_face_token = userdata.get("HF_TOKEN")

if not hugging_face_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it through the Colab Secrets panel."
    )

# Make the token available to DuckDB without displaying it
os.environ["HF_TOKEN"] = hugging_face_token

# Start the DuckDB connection
warehouse_connection = duckdb.connect()

# Allow DuckDB to retrieve the token from the environment
warehouse_connection.execute("""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        PROVIDER credential_chain
    )
""")

# Use March 2026 for development, not the final June sample
MARCH_2026_TABLE = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/'
    'fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("Warehouse connection configured successfully.")
print("Development month: March 2026")

Warehouse connection configured successfully.
Development month: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My provisional lane is **Refresh / Content Opportunity Scoring**.

For this assignment, one row represents **one pseudonymised content page belonging to one client on one reporting day**. Therefore, the unit of analysis is a **page-day**.

The primary warehouse table is `fact_content_daily_performance`, which contains daily search and analytics measurements for each content page.

I will use **1 March 2026 to 31 March 2026** as the development window. March 2026 is a mid-panel month, so it is suitable for testing the data contract and feature-building process without using the final month.

The planned output is a ranked review queue showing which pages an SEO specialist or content editor should investigate first for possible refresh, expansion, protection, pruning or monitoring.

I will deliberately exclude June 2026 and any future or label-derived information from the feature-building process. June 2026 is the final warehouse month and should remain a sealed test or outcome period rather than being used to develop the scoring logic.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Store the main data-contract settings in clearly named variables

lane_name = "Refresh / Content Opportunity Scoring"
unit_of_analysis = "One pseudonymised content page on one reporting day"
warehouse_table = "fact_content_daily_performance"
development_month = "2026-03"
sealed_test_month = "2026-06"
planned_output = "Ranked page-review queue"

print("Lane:", lane_name)
print("Unit of analysis:", unit_of_analysis)
print("Primary table:", warehouse_table)
print("Development month:", development_month)
print("Sealed test month:", sealed_test_month)
print("Planned output:", planned_output)


Lane: Refresh / Content Opportunity Scoring
Unit of analysis: One pseudonymised content page on one reporting day
Primary table: fact_content_daily_performance
Development month: 2026-03
Sealed test month: 2026-06
Planned output: Ranked page-review queue


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

For the **Refresh / Content Opportunity Scoring** lane, the fields will be separated into four groups.

### Features

The features will be measurable search-performance and engagement information that is genuinely available when a page receives its review-priority score.

I will select a maximum of five features after inspecting the actual warehouse schema. Possible feature categories include search impressions, clicks, average search position, click-through rate and engagement information.

Every selected feature must be knowable at the decision moment and must not contain future information.

### Label or proxy

The intended output is a score used to rank pages by review priority.

For this assignment, I will use a temporary proxy derived from an observed page-performance outcome. The final capstone should ideally use a future-window outcome, where earlier information is used to predict whether a page experiences a meaningful decline or opportunity later.

The proxy is decision-support evidence and does not prove that a page must be refreshed.

### Context fields

`report_date`, `client_id` and `content_id` will be retained as context fields.

They are needed to identify the reporting period, client and page, but they will not be used directly as predictive features because their values do not describe page quality or performance.

### Deliberately excluded fields

I will exclude:

* Any field used directly to create the label or proxy
* Information from the future outcome window
* June 2026 while developing the feature and label logic
* Identifiers such as `client_id` and `content_id` from the model features
* Analytics values from rows where the availability flag is not `TRUE`
* Any private or raw identifying information

These exclusions reduce the risk of target leakage, unavailable data and misleading model performance.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Inspect the real columns in the March 2026 warehouse table

schema_query = f"""
DESCRIBE
SELECT *
FROM {MARCH_2026_TABLE}
"""

warehouse_schema = warehouse_connection.execute(schema_query).df()

print("Number of columns in the warehouse table:", len(warehouse_schema))

display(
    warehouse_schema[
        ["column_name", "column_type"]
    ]
)


Number of columns in the warehouse table: 31


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


I will verify the data contract using exactly three queries on the March 2026 development partition.

The first query checks the grain by searching for duplicate combinations of `report_date`, `client_id` and `content_id`. If it returns no rows, the expected page-day grain holds for this slice.

The second query checks the size and date coverage of the slice. It reports the number of rows, clients and content pages, together with the earliest and latest reporting dates.

The third query checks analytics availability. It counts how many rows remain when `ga4_data_available IS TRUE` is applied. This is necessary because zero-filled analytics fields on unavailable rows must not be interpreted as genuine zero engagement.

After verifying these facts, I will build a five-feature page-level frame using only fields available at the decision moment. I will then deliberately introduce one label-derived feature, observe the inflated result, remove the leaked feature and retain the honest score.


In [6]:
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_at_expected_grain
FROM {MARCH_2026_TABLE}
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = warehouse_connection.execute(grain_query).df()

print("Duplicate page-day combinations found:", len(grain_check))

if grain_check.empty:
    print(
        "Grain check passed: one row per report date, "
        "client and content page."
    )
else:
    print("Grain check failed. Duplicate combinations are shown below.")

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate page-day combinations found: 0
Grain check passed: one row per report date, client and content page.


,report_date,client_hash_id,content_hash_id,rows_at_expected_grain


In [7]:
count_and_window_query = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM {MARCH_2026_TABLE}
"""

count_and_window_check = warehouse_connection.execute(
    count_and_window_query
).df()

print("March 2026 slice summary:")
display(count_and_window_check)

March 2026 slice summary:


,row_count,client_count,content_count,earliest_date,latest_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [8]:
availability_query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS rows_with_ga4_available,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NOT TRUE
    ) AS rows_without_ga4_available,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) / COUNT(*),
        2
    ) AS ga4_available_percentage

FROM {MARCH_2026_TABLE}
"""

availability_check = warehouse_connection.execute(
    availability_query
).df()

print("GA4 availability check:")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 availability check:


,total_rows,rows_with_ga4_available,rows_without_ga4_available,ga4_available_percentage
0,9841378,413966,9427412,4.21


The availability query returned **9,841,378 total rows** for March 2026. Only **413,966 rows, or 4.21%, had `ga4_data_available IS TRUE`**, while 9,427,412 rows did not have confirmed GA4 availability.

This means that GA4-based engagement features can only be used on a relatively small part of the March dataset. Zero-filled analytics values on unavailable rows must not be interpreted as genuine zero engagement. Therefore, any feature frame using GA4 measurements must filter to rows where `ga4_data_available IS TRUE`.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



A major limitation of my March 2026 data slice is the low availability of genuine GA4 engagement data.

The availability check showed that only **413,966 out of 9,841,378 rows, or 4.21%, had `ga4_data_available IS TRUE`**. This means approximately 95.79% of the rows did not have confirmed GA4 data available.

Therefore, analytics values such as sessions or engagement cannot be used safely across the entire March dataset. A zero value on a row where `ga4_data_available` is false may mean that the data was unavailable, rather than that the page received no visits or engagement.

Filtering to only GA4-available rows would produce a much smaller dataset and may make the analysis less representative of all clients and pages. It could overrepresent clients that adopted GA4 earlier or have more complete analytics histories.

For this reason, I will either use GA4 features only within the confirmed available subset or prioritise Search Console features that have broader coverage. Any findings will be described as observed and decision-support evidence rather than conclusions that apply equally to every client.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Use the result from the existing availability query
total_march_rows = int(
    availability_check.loc[0, "total_rows"]
)

ga4_available_rows = int(
    availability_check.loc[0, "rows_with_ga4_available"]
)

ga4_unavailable_rows = int(
    availability_check.loc[0, "rows_without_ga4_available"]
)

ga4_available_percentage = float(
    availability_check.loc[0, "ga4_available_percentage"]
)

ga4_unavailable_percentage = 100 - ga4_available_percentage

print("Named limitation: limited GA4 data availability")
print(f"Total March rows: {total_march_rows:,}")
print(f"Rows with confirmed GA4 data: {ga4_available_rows:,}")
print(f"Rows without confirmed GA4 data: {ga4_unavailable_rows:,}")
print(f"GA4 available: {ga4_available_percentage:.2f}%")
print(f"GA4 unavailable: {ga4_unavailable_percentage:.2f}%")

Named limitation: limited GA4 data availability
Total March rows: 9,841,378
Rows with confirmed GA4 data: 413,966
Rows without confirmed GA4 data: 9,427,412
GA4 available: 4.21%
GA4 unavailable: 95.79%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.